# **Imports**

In [1]:
import os
import glob
import pandas as pd
import geopandas as gpd
import numpy as np

# **Compiled Global Data**

# *Areas, Labels, and Patch Locations*

In [ ]:
######################################################################
# Compile local dataset area files into single global dataset files
######################################################################

# path to parent data directory
dataset_dir = r'../data'

# paths to local area directories
local_dataset_dirs = [d for d in os.listdir(dataset_dir) if os.path.isdir(os.path.join(dataset_dir, d)) and d != 'smoke']


##### get paths to local area/label/patch locations filees...
area_paths = []
label_paths = []
location_paths = []
for d in local_dataset_dirs:
    
    areas = glob.glob(f"{dataset_dir}/{d}/*areas.csv")
    area_paths.extend(areas)

    labels = glob.glob(f"{dataset_dir}/{d}/*labels.csv")
    label_paths.extend(labels)

    locations = glob.glob(f"{dataset_dir}/{d}/*patches.geojson")
    location_paths.extend(locations)


##### combine files and save to single files in parent directory...
areas = [pd.read_csv(path) for path in area_paths]
df_areas = pd.concat(areas, ignore_index=True)
df_areas.to_csv(f"{dataset_dir}/earthscape_areas.csv", index=False)

labels = [pd.read_csv(path) for path in label_paths]
df_labels = pd.concat(labels, ignore_index=True)
df_labels.to_csv(f"{dataset_dir}/earthscape_labels.csv", index=False)

locations = [gpd.read_file(path) for path in location_paths]
gdf_locations = pd.concat(locations, ignore_index=True)
gdf_locations.to_file(f"{dataset_dir}/earthscape_patches.geojson", driver='GeoJSON')


## *Patch Normalization Statistics*

In [ ]:
# path to parent data directory
dataset_dir = r'../data'

# paths to local area directories
local_dataset_dirs = [d for d in os.listdir(dataset_dir) if os.path.isdir(os.path.join(dataset_dir, d)) and d != 'smoke']

df_all = pd.DataFrame()

for d in local_dataset_dirs:

    working_dir = f"{dataset_dir}/{d}"
    stats_path = glob.glob(f"{working_dir}/*_stats.csv")[0]
    df = pd.read_csv(stats_path)
    df['dataset'] = d
    num_patches = len(glob.glob(f"{working_dir}/patches_{d}/*.tif")) / 38
    df['n'] = num_patches * (256 * 256)
    df_all = pd.concat([df_all, df], ignore_index=True)


df_all['n_mu'] = df_all['n'] * df_all['mean']

N = df_all.groupby('channel')['n'].sum()
sum_nmu = df_all.groupby('channel')['n_mu'].sum()
mu_global = sum_nmu / N

df_all = df_all.merge(mu_global.rename('global_mean'), on='channel')

within = ((df_all['n'] - 1) * (df_all['sd'] ** 2)).groupby(df_all['channel']).sum()
between = (df_all['n'] * (df_all['mean'] - df_all['global_mean']) ** 2).groupby(df_all['channel']).sum()
var_global = (within + between) / (N - 1)
sd_global = np.sqrt(var_global)

global_stats = pd.DataFrame({'channel': N.index, 'mean': mu_global, 'sd': sd_global})

output_path = f"{dataset_dir}/earthscape_image_stats.csv"
global_stats.to_csv(output_path, index=False)